# Parts 3–4 · 00 Setup and access check

This notebook runs once before any download. It:

1. records the environment (Python and package versions, for the reproducibility checklist);
2. connects to WRDS with **your own** account;
3. checks access to every table parts 3–4 need and records their exact column names;
4. builds the trading calendar from CRSP;
5. rebuilds the provisional release clock on that calendar.

**Before running:** clone the repository, open this notebook inside it, and have Duo Mobile on your phone.

**Folder layout**

```
S08 NLP/3_4_surprise_returns/
  code/     common.py, clock.py; later download, surprise, returns, funnel
  cache/    raw WRDS pulls, one file per table and year (WRDS-derived: never in the submission)
  data/     built tables keyed by accession
  results/  tables and figures for the report
  logs/     access reports and run logs
```

In [3]:
!pip install -q wrds pandas_market_calendars pyarrow
# If Colab asks you to restart the session after this install, restart it
# and continue from the next cell (no need to rerun this one).
# A red pip message about ibis-framework and toolz is harmless: ibis is a
# Colab pre-install we never use.

In [5]:
import os, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Optional private data root; code always comes from this checkout.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
os.environ["TEAM_DIR"] = str(TEAM_DIR)
SCRIPTS = REPO_ROOT / "scripts"
NOV2_DATA = TEAM_DIR / "1_2nov" / "nov2_data"
if not NOV2_DATA.is_dir():
    NOV2_DATA = TEAM_DIR / "files csv"



PROJECT_DIR = TEAM_DIR / "3_4_surprise_returns"
MAHAUT_DATA = NOV2_DATA   # only used by the optional clock cell



for sub in ["cache", "data", "results", "logs"]:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
sys.path.insert(0, str(REPO_ROOT / "3_4_surprise_returns" / "code"))

missing = [f for f in ["common.py", "clock.py"] if not (REPO_ROOT / "3_4_surprise_returns" / "code" / f).exists()]
assert not missing, f"Missing code in {PROJECT_DIR / 'code'}: {missing}"

print("team folder:   ", TEAM_DIR)
print("project folder:", PROJECT_DIR)
print("code files:    ", sorted(os.listdir(REPO_ROOT / "3_4_surprise_returns" / "code")))

Mounted at /content/drive
team folder:    /content/drive/MyDrive/S08 NLP
project folder: /content/drive/MyDrive/S08 NLP/3_4_surprise_returns
code files:     ['clock.py', 'common.py']


In [6]:
# Environment record: goes into the access report and, later, the README.
import json, platform, datetime as dt
import importlib.metadata as md

env = {"run_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version()}
for pkg in ["wrds", "pandas", "numpy", "sqlalchemy", "psycopg2-binary",
            "pandas_market_calendars", "pyarrow"]:
    try:
        env[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        env[pkg] = "not installed"
print(json.dumps(env, indent=2))

{
  "run_at_utc": "2026-11-04T14:28:23+00:00",
  "python": "3.13.18",
  "wrds": "3.5.0",
  "pandas": "2.2.3",
  "numpy": "2.4.6",
  "sqlalchemy": "2.0.57",
  "psycopg2-binary": "2.9.13",
  "pandas_market_calendars": "5.5.0",
  "pyarrow": "23.0.1"
}


## Connect to WRDS

Use your own account: WRDS accounts are personal, and entitlements belong to the account, so Mahaut's check does not cover yours. You will be asked for your password; then approve the Duo push on your phone within a few minutes.

If asked whether to create a `.pgpass` file, answer **n**. Never put your password in the notebook or in Drive.

In [9]:
import wrds

WRDS_USER = input("WRDS username: ").strip()
db = wrds.Connection(wrds_username=WRDS_USER)

WRDS username: bblank
Enter your WRDS username [bblank]:bblank
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


## Access and schema check

Each line selects one row from the table itself, because a table can appear in the catalogue and still refuse a query. It also records every column name, so the download queries are written against the real schema rather than from memory.

In [10]:
import pandas as pd

CHECKS = [
    # (label, table, optional coverage query)
    ("CRSP daily (CIZ)", "crsp.dsf_v2",
     "select min(dlycaldt) as first, max(dlycaldt) as last from crsp.dsf_v2 where permno = 14593"),
    ("CRSP names (CIZ)", "crsp.stocknames_v2", None),
    ("CRSP delistings (CIZ)", "crsp.stkdelists", None),
    ("CCM link history", "crsp.ccmxpf_lnkhist",
     "select linktype, linkprim, count(*) as n from crsp.ccmxpf_lnkhist group by 1, 2 order by 3 desc"),
    ("Fama-French daily factors", "ff.factors_daily",
     "select min(date) as first, max(date) as last from ff.factors_daily"),
    ("I/B/E/S actuals, unadjusted", "ibes.actu_epsus",
     "select pdicity, measure, count(*) as n, max(anndats) as last from ibes.actu_epsus "
     "where anndats >= '2024-01-01' group by 1, 2 order by 3 desc"),
    ("I/B/E/S detail, unadjusted", "ibes.detu_epsus",
     "select measure, fpi, count(*) as n, max(anndats) as last from ibes.detu_epsus "
     "where anndats >= '2025-01-01' group by 1, 2 order by 3 desc"),
    ("I/B/E/S guidance", "ibes.det_guidance",
     "select measure, count(*) as n, max(anndats) as last from ibes.det_guidance "
     "where anndats >= '2025-01-01' group by 1 order by 2 desc"),
    ("I/B/E/S to CRSP link", "wrdsapps.ibcrsphist",
     "select score, count(*) as n from wrdsapps.ibcrsphist group by 1 order by 1"),
    ("SEC CIK to GVKEY link", "wrdssec.wciklink_gvkey", None),
    ("Compustat quarterly (GAAP EPS check)", "comp.fundq",
     "select max(datadate) as last from comp.fundq"),
]


def rollback():
    # A failed query leaves the connection in an aborted transaction;
    # without this, every later check would fail too.
    try:
        db.connection.rollback()
    except Exception:
        pass


report = {"env": env, "user": WRDS_USER, "tables": {}}
for label, table, cov_sql in CHECKS:
    entry = {"label": label}
    try:
        one = db.raw_sql(f"select * from {table} limit 1")
        entry["access"] = "ok"
        entry["columns"] = list(one.columns)
    except Exception as e:
        rollback()
        entry["access"] = "DENIED_OR_MISSING"
        entry["error"] = str(e).splitlines()[0][:200]
    if cov_sql and entry["access"] == "ok":
        try:
            entry["coverage"] = db.raw_sql(cov_sql).astype(str).to_dict("records")
        except Exception as e:
            rollback()
            entry["coverage_error"] = str(e).splitlines()[0][:200]
    report["tables"][table] = entry
    print(f"{entry['access']:<18} {table:<26} {label}")

ok                 crsp.dsf_v2                CRSP daily (CIZ)
ok                 crsp.stocknames_v2         CRSP names (CIZ)
ok                 crsp.stkdelists            CRSP delistings (CIZ)
ok                 crsp.ccmxpf_lnkhist        CCM link history
ok                 ff.factors_daily           Fama-French daily factors
ok                 ibes.actu_epsus            I/B/E/S actuals, unadjusted
ok                 ibes.detu_epsus            I/B/E/S detail, unadjusted
ok                 ibes.det_guidance          I/B/E/S guidance
ok                 wrdsapps.ibcrsphist        I/B/E/S to CRSP link
ok                 wrdssec.wciklink_gvkey     SEC CIK to GVKEY link
ok                 comp.fundq                 Compustat quarterly (GAAP EPS check)


In [11]:
# Table names we may need but have not confirmed (revenue estimates,
# stopped estimates, other CIZ files). Listing them lets us pick the right ones.
def tables_like(library, *patterns):
    try:
        names = db.list_tables(library=library)
    except Exception as e:
        rollback()
        return [f"error: {str(e).splitlines()[0][:120]}"]
    return sorted(n for n in names if any(p in n for p in patterns))


report["discovery"] = {
    "ibes": tables_like("ibes", "epsus", "xeps", "guid", "stop", "sum", "act", "det"),
    "crsp": tables_like("crsp", "_v2", "delist", "names", "ccm"),
    "ff": tables_like("ff", "factor"),
}

out = PROJECT_DIR / "logs" / f"access_report_{WRDS_USER}_{dt.date.today()}.json"
out.write_text(json.dumps(report, indent=2, default=str))

# Compact printout: copy everything from here down and send it back.
print("=" * 70)
print(json.dumps(env))
for table, e in report["tables"].items():
    print(f"\n{table}: {e['access']}")
    if "columns" in e:
        print("  columns:", ", ".join(e["columns"]))
    if "coverage" in e:
        print("  coverage:", e["coverage"][:8])
    if "error" in e or "coverage_error" in e:
        print("  error:", e.get("error") or e.get("coverage_error"))
for lib, names in report["discovery"].items():
    print(f"\n{lib} tables ({len(names)}):", ", ".join(names))
print("\nsaved:", out)

{"run_at_utc": "2026-11-04T14:28:23+00:00", "python": "3.13.18", "wrds": "3.5.0", "pandas": "2.2.3", "numpy": "2.4.6", "sqlalchemy": "2.0.57", "psycopg2-binary": "2.9.13", "pandas_market_calendars": "5.5.0", "pyarrow": "23.0.1"}

crsp.dsf_v2: ok
  columns: permno, hdrcusip, permco, siccd, nasdissuno, yyyymmdd, sharetype, securitytype, securitysubtype, usincflg, issuertype, primaryexch, conditionaltype, tradingstatusflg, dlycaldt, dlydelflg, dlyprc, dlyprcflg, dlycap, dlycapflg, dlyprevprc, dlyprevprcflg, dlyprevdt, dlyprevcap, dlyprevcapflg, dlyret, dlyretx, dlyreti, dlyretmissflg, dlyretdurflg, dlyorddivamt, dlynonorddivamt, dlyfacprc, dlydistretflg, dlyvol, dlyclose, dlylow, dlyhigh, dlybid, dlyask, dlyopen, dlynumtrd, dlymmcnt, dlyprcvol, dlycumfacpr, dlycumfacshr, cusip, ticker, exchangetier, shrout
  coverage: [{'first': '1980-12-12', 'last': '2025-12-31'}]

crsp.stocknames_v2: ok
  columns: permno, permco, namedt, nameenddt, securitybegdt, securityenddt, hdrcusip, hdrcusip9, cusi

## Trading calendar

Trading days come from CRSP itself, cross-checked against the NYSE calendar, which also supplies early closes (13:00) and the sessions after CRSP's last date. Both "days in one but not the other" counts should be 0.

In [12]:
from common import build_trading_calendar

cal = build_trading_calendar(db)
cal.to_csv(PROJECT_DIR / "data" / "trading_calendar.csv", index=False)
print(cal.tail(3).to_string())

trading days per year: {2005: 252, 2006: 251, 2007: 251, 2008: 253, 2009: 252, 2010: 252, 2011: 252, 2012: 250, 2013: 252, 2014: 252, 2015: 252, 2016: 252, 2017: 251, 2018: 251, 2019: 252, 2020: 253, 2021: 252, 2022: 251, 2023: 250, 2024: 252, 2025: 250}
days in NYSE calendar but not CRSP: 0 []
days in CRSP but not NYSE calendar: 0 []
early closes: 46
extended past CRSP's last day with 251 NYSE sessions
           date close_et                    source
5531 2026-12-29    16:00  nyse_calendar_after_crsp
5532 2026-12-30    16:00  nyse_calendar_after_crsp
5533 2026-12-31    16:00  nyse_calendar_after_crsp


## Provisional release clock on the real calendar

Same rule as before: wire time first, I/B/E/S only as the second source (or a development stand-in), nothing from the EDGAR submissions JSON. Expected: about 115,976 events, about 79,000 with two sources agreeing.

In [13]:
from common import load_trading_calendar
from clock import build_provisional_clock, validate_clock, summarise

cal = load_trading_calendar(PROJECT_DIR / "data" / "trading_calendar.csv")
clk = build_provisional_clock(str(MAHAUT_DATA / "three_source_timing.csv"),
                              str(MAHAUT_DATA / "event_table.csv"), cal)
validate_clock(clk, cal)
print(summarise(clk).to_string())
clk.to_csv(PROJECT_DIR / "data" / "clock_provisional.csv", index=False)
print("saved:", PROJECT_DIR / "data" / "clock_provisional.csv")

timing rows 117,926: dropped 1,949 with no accession, 1 duplicate accession(s)
clock validation passed
                     events  share_pct
status                                
two_source_agree      79094       68.2
single_source_ibes    24771       21.4
no_clock               9086        7.8
single_source          2219        1.9
two_source_conflict     806        0.7
saved: /content/drive/MyDrive/S08 NLP/3_4_surprise_returns/data/clock_provisional.csv


In [14]:
db.close()